# RQ6 — Robustness (CV and perturbation)

**Research question (RQ6).** How stable is performance under repeated cross-validation and mild numeric feature noise?

**Task:** regression to predict `revenue_million`. **Outputs:** CSV + PDF in `./outputs`.

## Methodology (this notebook)

- 5-fold CV RMSE for Ridge, Random Forest, Gradient Boosting.
- Perturbation test: add 10% Gaussian noise (relative to column std) to numeric inputs and evaluate Random Forest on a held-out split.

In [1]:
# Setup: paths, load data, modeling frame (Global movies — regression)
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = os.path.exists("/kaggle/input")
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path(".")
OUT = Path("/kaggle/working") if IS_KAGGLE else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)
RQ_PREFIX = "RQ06"
RANDOM_STATE = 42
TARGET = "revenue_million"

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)


def find_raw_table_path() -> Path:
    preferred = ("global_movies_dataset_1950_2026.csv",)
    found: list[Path] = []
    if IS_KAGGLE:
        for root, _, files in os.walk(INPUT_ROOT):
            for fn in files:
                p = Path(root) / fn
                if p.suffix.lower() in {".csv"}:
                    found.append(p)
    else:
        for p in INPUT_ROOT.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".csv"}:
                found.append(p)
    for name in preferred:
        for p in found:
            if p.name.lower() == name.lower():
                return p
    if found:
        return found[0]
    raise FileNotFoundError(
        "No CSV found. Add the dataset via Kaggle Add Input or place global_movies_dataset_1950_2026.csv next to this notebook."
    )


def prepare_modeling_df(raw: pd.DataFrame) -> pd.DataFrame:
    d = raw.copy()
    numeric_cols = [
        "release_year",
        "runtime_min",
        "imdb_rating",
        "votes",
        "budget_million",
        "marketing_budget_million",
        "metascore",
        "audience_score",
        "award_nominations",
        "award_wins",
    ]
    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    if "franchise_flag" in d.columns:
        d["franchise_flag"] = pd.to_numeric(d["franchise_flag"], errors="coerce")

    cat_cols = [
        "genre",
        "subgenre",
        # high-cardinality fields intentionally excluded for speed
        "country",
        "language",
        "streaming_platform",
    ]
    for c in cat_cols:
        if c in d.columns:
            d[c] = d[c].fillna("missing").astype(str)

    d[TARGET] = pd.to_numeric(d[TARGET], errors="coerce")
    d = d.dropna(subset=[TARGET])
    if len(d) > 30000:
        d = d.sample(12000, random_state=RANDOM_STATE)
    return d


def regression_metrics(y_true, y_pred) -> dict:
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "R2": float(r2_score(y_true, y_pred)),
    }


RAW_PATH = find_raw_table_path()
df = pd.read_csv(RAW_PATH)
MODEL_DF = prepare_modeling_df(df)

DEFAULT_FEATURE_COLS = [
    "release_year",
    "runtime_min",
    "imdb_rating",
    "votes",
    "budget_million",
    "marketing_budget_million",
    "metascore",
    "audience_score",
    "award_nominations",
    "award_wins",
    "franchise_flag",
    "genre",
    "subgenre",
    "country",
    "language",
    "streaming_platform",
]
LEAKY_OR_LABEL_COLS = {"roi_pct", "top_100_prob", "blockbuster_flag"}
FEATURE_COLS = [c for c in DEFAULT_FEATURE_COLS if c in MODEL_DF.columns and c not in LEAKY_OR_LABEL_COLS]

print("Loaded:", RAW_PATH)
print("Rows (modeling):", len(MODEL_DF), "Features:", len(FEATURE_COLS))


Loaded: global_movies_dataset_1950_2026.csv
Rows (modeling): 12000 Features: 16


In [2]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = MODEL_DF[FEATURE_COLS]
y = MODEL_DF[TARGET]

from pandas.api.types import is_numeric_dtype

cat_cols = [c for c in FEATURE_COLS if not is_numeric_dtype(MODEL_DF[c])]
num_cols = [c for c in FEATURE_COLS if is_numeric_dtype(MODEL_DF[c])]


def make_preprocessor():
    num_pipe = Pipeline(
        [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    )
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)]
    )


pipelines = {
    "Ridge": Pipeline([("prep", make_preprocessor()), ("model", Ridge(random_state=RANDOM_STATE, alpha=2.0))]),
    "RandomForest": Pipeline(
        [
            ("prep", make_preprocessor()),
            (
                "model",
                RandomForestRegressor(
                    random_state=RANDOM_STATE,
                    n_estimators=80,
                    min_samples_leaf=10,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "GradientBoosting": Pipeline(
        [
            ("prep", make_preprocessor()),
            (
                "model",
                GradientBoostingRegressor(
                    random_state=RANDOM_STATE,
                    max_depth=4,
                    n_estimators=140,
                    learning_rate=0.08,
                ),
            ),
        ]
    ),
}

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipe in pipelines.items():
    scores = cross_val_score(
        clone(pipe), X, y, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    rmse = -scores
    rows.append({"Model": name, "CV_RMSE_mean": rmse.mean(), "CV_RMSE_std": rmse.std()})

tbl = pd.DataFrame(rows)
tbl.to_csv(OUT / f"{RQ_PREFIX}_table_cv_rmse.csv", index=False)

# Perturbation: add Gaussian noise to numeric columns in raw frame
rng = np.random.default_rng(0)
Xn = X.copy()
for c in num_cols:
    sigma = Xn[c].std() * 0.10
    Xn[c] = Xn[c] + rng.normal(0, sigma, size=len(Xn))

X_train, X_test, y_train, y_test = train_test_split(
    Xn, y, test_size=0.25, random_state=RANDOM_STATE
)
pipe = clone(pipelines["RandomForest"])
pipe.fit(X_train, y_train)
noise_metrics = regression_metrics(y_test, pipe.predict(X_test))

base_train, base_test, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)
pipe0 = clone(pipelines["RandomForest"])
pipe0.fit(base_train, ytr)
base_metrics = regression_metrics(yte, pipe0.predict(base_test))

pert = pd.DataFrame(
    [
        {"Scenario": "standard_split_RF", **base_metrics},
        {"Scenario": "RF_with_10pct_numeric_noise", **noise_metrics},
    ]
)
pert.to_csv(OUT / f"{RQ_PREFIX}_table_noise_perturbation_rf.csv", index=False)

fig, ax = plt.subplots(figsize=(6.8, 4.8))
sns.barplot(data=tbl, x="Model", y="CV_RMSE_mean", ax=ax, palette="muted")
ax.errorbar(
    x=np.arange(len(tbl)),
    y=tbl["CV_RMSE_mean"],
    yerr=tbl["CV_RMSE_std"],
    fmt="none",
    c="black",
    capsize=4,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
ax.set_title("RQ6 — 5-fold CV RMSE (mean ± std)")
plt.tight_layout()
fig.savefig(OUT / f"{RQ_PREFIX}_fig_cv_rmse_bar.pdf")
plt.close()

tbl, pert


(              Model  CV_RMSE_mean  CV_RMSE_std
 0             Ridge    155.722322     4.363918
 1      RandomForest    152.649168     4.344682
 2  GradientBoosting    154.982235     4.428610,
                       Scenario        MAE        RMSE        R2
 0            standard_split_RF  80.276164  144.582620  0.400757
 1  RF_with_10pct_numeric_noise  80.943175  144.512582  0.401337)